> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Tracking and reproducing results

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/main/docs/user_guide/09_how_results_are_computed.ipynb)

You specify a mathematical request, such as an expectation or a conditional distribution.
ProbPipe selects a computation from the representations and capabilities of the inputs.
Use this chapter to inspect that selection, control approximation, and reproduce a result.
The examples extend the basic calling behavior introduced in [Key concepts](01_key_concepts.ipynb).

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab's preinstalled PyMC 5 does not work with ArviZ 1.x, so the install upgrades PyMC.
    %pip install --quiet "probpipe-core[pymc] @ git+https://github.com/TARPS-group/prob-pipe.git@main"

In [2]:
import jax
import jax.numpy as jnp
import numpy as np

from probpipe import (
    Normal,
    NumericArray,
    NumericArraySpec,
    ResolutionError,
    condition_on,
    conditional_distribution,
    config,
    function,
    log_prob,
    mean,
    operation_registry,
    prob,
    provenance_ancestors,
    replay_run,
    sample,
    workflow_run,
)

## From a request to a result

A call connects the mathematical objects you supply to the computation that produces the result:

| Part of a call | What it means for the user |
|---|---|
| Inputs | Supply values, distributions, or batches appropriate to the computation. |
| Validation and planning | ProbPipe checks the arguments and any declared input and output specs. |
| Lifting | A wrapped function applied to a distribution where it expects a value computes the distribution of its output. A batch argument can instead produce a sweep of evaluations. |
| Selection | Operations choose routes, and lifted functions choose evaluation rules. Registries supply methods such as inference algorithms. |
| Execution | ProbPipe evaluates the selected computation, compiling or mapping it when the chosen execution mode permits. |
| Result | The returned term carries a label and, with provenance enabled, a record of its computation. |

For example, `mean(prior)` asks for an expectation.
A mean capability lets ProbPipe evaluate that expectation directly.
A sampling representation can instead support an approximation from draws.
The mathematical request is the same; the available representation determines its computation.

Start by inspecting the selection report.
Then use controls to choose the computation and a seeded run to reproduce it.
The later sections explain the result's provenance, the labels that name each result, and why a term cannot change after it is made.

## Routes

We use one small model throughout.
The prior of a mean $\mu$ is a standard normal distribution, and given $\mu$, five measurements $y$ are independent normal draws with mean $\mu$ and standard deviation 1.

In [3]:
prior = Normal("mu", 0.0, 1.0, label="prior")


def y_given_mu(mu: jax.Array) -> Normal:
    return Normal("y", mu * jnp.ones(5), 1.0)


likelihood = conditional_distribution(
    y_given_mu, label="likelihood", given_spec=prior.event_spec
)
model = (likelihood & prior).with_label("model")
y = jnp.array([1.2, 0.4, 2.1, 1.5, 0.9])
print("outputs of the model:", list(model.event_spec.outputs))

outputs of the model: ['y', 'mu']


Each operation has a list of **routes**, which are its implementations, and a call selects the first route that applies.
An operation's `check` takes the arguments of a call and reports the route the call would take, without computing anything.
A normal distribution has a formula for its mean, so `mean` takes the exact route `closed_form`:

In [4]:
report = mean.check(prior)
print("route for the mean of the prior:", report.route)
print("exact:", report.exact)

route for the mean of the prior: closed_form
exact: True


The model is a joint distribution built from a conditional distribution, and this representation supplies no whole-model mean capability.
The Gaussian model has analytic means, but the operation uses the routes that the implementation provides.
Displayed in a notebook, the report shows every route that `mean` considered and the reason each one declined:

In [5]:
mean.check(model)

,route,method,exact,feasible,reason
,closed_form,,exact,infeasible,FactoredDistribution 'model' does not implement SupportsMean
,by_output,exact,exact,infeasible,"the outputs ('y',) have no exact mean, but ('mu',) do"
selected,by_output,approximate,approximate,feasible,


The route `by_output` computes the mean of each output through its view.
The mean of `mu` takes the exact route `closed_form`, and the mean of `y` takes the route `monte_carlo`, which averages draws from the model, so the result is approximate.
The registry of operations, `operation_registry`, lists each operation with its operands and its routes in the order a call tries them:

In [6]:
print(operation_registry.describe("mean"))

mean(d, given) — primitive
  The mean ``𝔼[X]`` for ``X ~ d``, a value shaped like one draw.
  operands:
    d: DistributionSpec, ConditionalDistributionSpec; read by the result rule
    given: RecordSpec
  routes, in selection order:
    closed_form (capability, exact; requires SupportsMean): membership in SupportsMean suffices
    by_output (structural, exactness delegated): the event is a record of several outputs, at least one of which has an exact route of its own, and the rest of which have a route together
    monte_carlo (fallback, approximate): The law samples numeric draws, which average coordinatewise, or laws, which average to a mixture.


`operation_registry.list()` returns one summary per operation, and its `name` attribute gives the names of all the operations:

In [7]:
print("operations:", ", ".join(summary.name for summary in operation_registry.list()))

operations: convert, sample, condition_on, log_prob, unnormalized_log_prob, prob, unnormalized_prob, random_log_prob, random_unnormalized_log_prob, inverse, log_det_jacobian, joint, marginal, factor, pushforward, mixture, mean, variance, standard_deviation, cov, quantile, expectation


## Choosing a route with options

An operation's `using` returns a copy of the operation with revised controls, and calling the copy applies them.
The control `method` names the route to take.
The mean of the prior is 0, and the route `monte_carlo` estimates it from draws:

In [8]:
with workflow_run(seed=0):
    estimate = mean.using(method="monte_carlo")(prior)
    finer_estimate = mean.using(method="monte_carlo", num_samples=100_000)(prior)
print("mean of mu from 1,024 draws:", round(float(estimate.raw), 4))
print("mean of mu from 100,000 draws:", round(float(finer_estimate.raw), 4))

mean of mu from 1,024 draws: -0.0063
mean of mu from 100,000 draws: -0.0061


The control `num_samples` sets the number of draws, which is 1,024 by default, and increasing it generally reduces Monte Carlo error.
An individual estimate can still move farther from 0.
For `condition_on`, `method` also names an inference method, such as `blackjax_nuts`, as [Fitting a model to data](06_conditioning_and_inference.ipynb) shows.

The control `exact_only=True` refuses every approximate route.
The mean of `y` has no exact route, so `check` reports that the call is infeasible, and the call raises a `ResolutionError` that names the exact call on the view of `mu`:

In [9]:
exact_mean = mean.using(exact_only=True)
print("feasible with exact_only:", exact_mean.check(model).feasible)
try:
    exact_mean(model)
except ResolutionError as error:
    print("error:", error)

feasible with exact_only: False
error: mean: the outputs ('y',) have no exact mean, but ('mu',) do; call mean on the view of each output that does, as mean(d['mu']). Routes tried with exact_only: closed_form: FactoredDistribution 'model' does not implement SupportsMean; by_output (exact): the outputs ('y',) have no exact mean, but ('mu',) do; call mean on the view of each output that does, as mean(d['mu'])


A copy keeps the controls set on it, and `options` reads them.
The original operation is unchanged, and giving a control as `None` resets it to its default:

In [10]:
by_draws = mean.using(method="monte_carlo")
print("method of mean:", mean.options["method"])
print("method of the copy:", by_draws.options["method"])
print("method after the reset:", by_draws.using(method=None).options["method"])

method of mean: None
method of the copy: monte_carlo
method after the reset: None


A control set on an operation also governs the operations that its computation calls.
`prob` computes the density from `log_prob`, so `prob.using(exact_only=True)` restricts that `log_prob` to exact routes, and the density of the prior is exact:

In [11]:
density = prob.using(exact_only=True)(prior, 0.0)
print("density of the prior at 0:", density.raw)
print("exact:", density.provenance.metadata["exact"])

density of the prior at 0: 0.3989423
exact: True


A function made with `@function` takes the same controls.
Called on the normal distribution below, it is evaluated at `max_evaluations` draws of its argument, as [Propagating uncertainty through functions](03_functions_and_lifting.ipynb) explains.
For one new measurement $y_{\text{new}} \sim \text{Normal}(\mu, 1)$, the probability that it is positive is $\Phi(\mu)$:

In [12]:
@function
def prob_next_positive(mu: jax.Array) -> jax.Array:
    return jax.scipy.stats.norm.cdf(mu)


with workflow_run(seed=0):
    default_draws = prob_next_positive(prior)
    more_draws = prob_next_positive.using(max_evaluations=2000)(prior)
print("draws at the default count:", default_draws.num_atoms)
print("draws with max_evaluations=2000:", more_draws.num_atoms)

draws at the default count: 256
draws with max_evaluations=2000: 2000


## Reproducible draws

Operations take no random key.
Instead, `workflow_run(seed=...)` seeds every random draw inside its block, so two blocks with the same seed and the same calls give the same draws:

In [13]:
with workflow_run(seed=0):
    first = sample(prior, sample_shape=3)
    second = sample(prior, sample_shape=3)
with workflow_run(seed=0):
    first_again = sample(prior, sample_shape=3)
print("first draws:", first.raw)
print("second draws in the same block:", second.raw)
print("first draws of the rerun:", first_again.raw)

first draws: [-0.40515503 -1.366459   -0.9853808 ]
second draws in the same block: [-0.03004714 -0.0036601  -0.02555274]
first draws of the rerun: [-0.40515503 -1.366459   -0.9853808 ]


The draws of a call follow from the seed and the call's position in the block.
Two calls in one block therefore get different draws, and a rerun of the block repeats both.
A call outside every `workflow_run` block draws from a fresh seed each time.
A random draw inside a `jax.jit` or `jax.vmap` that our own code opens raises `RuntimeError`, so we call the random operations outside such a transformation.

`replay_run` reproduces one recorded call from its result's provenance, which the next section describes.
We run the same call inside it, and it repeats the recorded draws:

In [14]:
with replay_run(second.provenance):
    replayed = sample(prior, sample_shape=3)
print("replayed draws:", replayed.raw)

replayed draws: [-0.03004714 -0.0036601  -0.02555274]


The provenance stores the seed and the position of the call, so the replay needs no `workflow_run` block.
The replay checks that the call matches the recorded one, and a `ReplayCompatibilityError` reports a call that does not.

## Provenance

Every result records how it was computed in its **provenance**.
We condition the model on the measurements, which takes an approximate route because no implemented exact route applies to this representation of the model:

In [15]:
with workflow_run(seed=0):
    posterior = condition_on(model, {"y": y})

record = posterior.provenance
print("label of the posterior:", posterior.label)
print("notation of the posterior:", posterior)
print("route:", record.metadata["route"])
print("method:", record.metadata["method"])
print("exact:", record.metadata["exact"])
print("computation history:")
print(record)

label of the posterior: model
notation of the posterior: model(mu; y)
route: inference_methods
method: blackjax_nuts
exact: False
computation history:
1. model = with_label(likelihood·prior)
2. condition_on(1, given)                route inference_methods, method blackjax_nuts, approximate


The printed provenance lists the recorded computation steps and the selected routes or methods.
Steps internal to an inference route are folded into the call that ran them.
The full record still retains them for inspection.

The provenance's `metadata` holds the route, the inference method that route selected, and whether the result is exact.
The posterior keeps the model's label.
Its notation `model(mu; y)` lists the output `mu` and the conditioned path `y` after the semicolon.

The provenance's `parents` are the terms the operation consumed:

In [16]:
for parent in record.parents:
    print(f"parent: {parent.label} ({parent.type_name})")

parent: condition_on (Operation)
parent: model (FactoredDistribution)
parent: model (EmpiricalDistribution)


The parents record three inputs to this result:

- the operation `condition_on`;
- the model being conditioned;
- the empirical distribution returned by `blackjax_nuts`, from which `condition_on` built the posterior.
A result computed from the posterior records its own step.
The mean of the posterior holds one field per output, under the output's own name, so `posterior_mean["mu"]` selects the mean of `mu`:

In [17]:
posterior_mean = mean(posterior)
print("posterior mean:", repr(posterior_mean))
print("posterior mean of mu:", round(float(posterior_mean["mu"].raw), 3))
print("route of the mean:", posterior_mean.provenance.metadata["route"])
print("exact:", posterior_mean.provenance.metadata["exact"])

posterior mean: NumericRecord('𝔼[mu ~ model; y]')
  mu   1.0252991
posterior mean of mu: 1.025
route of the mean: closed_form
exact: True


The mean is labeled `𝔼[mu ~ model; y]`, which is the expectation of a draw of `mu` from the posterior.
It is exact for the 4,000 draws of the posterior, since the mean of a distribution on finitely many points is a weighted sum.
The approximation entered one step earlier, when the sampler drew them.
For this normal model the posterior mean is $\sum_i y_i / 6 \approx 1.017$, so the sampler's draws account for the difference of about 0.01.

`provenance_ancestors` follows the parents back through every step and returns a descriptor of each ancestor, whose `provenance` is the record of the step that produced it.
Here the forecast evaluates `prob_next_positive` at draws from the posterior, and its mean summarizes the result:

In [18]:
with workflow_run(seed=1):
    forecast = prob_next_positive(posterior["mu"])
forecast_mean = mean(forecast)

print("forecast:", forecast)
print("label of its mean:", forecast_mean.label)

print("P(next measurement > 0), posterior mean:", round(float(forecast_mean.raw), 3))
steps = [forecast_mean.provenance] + [
    ancestor.provenance for ancestor in provenance_ancestors(forecast_mean) if ancestor.provenance
]
for step in steps:
    if "exact" in step.metadata:
        metadata = step.metadata
        print(f"{metadata['func']}: route {metadata['route']}, exact {metadata['exact']}")

forecast: prob_next_positive(mu ~ model; y)
label of its mean: 𝔼[prob_next_positive(mu ~ model; y)]
P(next measurement > 0), posterior mean: 0.814
mean: route closed_form, exact True
prob_next_positive: route sampling_lift, exact False
condition_on: route inference_methods, exact False


The steps print from the last back to the first.
`mean` is exact for the forecast's draws.
The two steps before it are approximate: the evaluation rule `sampling_lift` evaluated `prob_next_positive` at 256 draws of the posterior, and `blackjax_nuts` drew the posterior.
Both approximate steps can contribute to the forecast's Monte Carlo error.
The forecast displays as `prob_next_positive(mu ~ model; y)`, which is the function applied to a draw of `mu` from the posterior, and its mean is labeled as the expectation of that forecast.
`provenance_dag` builds the same chain as a graph, which the graphviz package draws.

## Sample counts and execution

[Propagating uncertainty through functions](03_functions_and_lifting.ipynb) explains `max_evaluations` and its process-wide default.
An inference method has its own budget, such as `num_results` and `num_chains`, set through `method_options`.
A sample-count control changes computational effort; it does not change the intended distribution.

Ordinary calls execute in the current process by default.
[Ray via Prefect](ray.md) describes opting into task orchestration when independent evaluations are expensive enough to distribute.

An **executor** determines how a set of evaluations runs, such as one at a time or on worker threads.
The `executor` control selects that strategy when ordinary Python evaluation is needed.
The function below converts its input to a Python number, so it is declared opaque to JAX with `traceable=False`.
Its sequential and threaded executions should produce matching atoms for the same seed and call:


In [19]:
@function(traceable=False)
def magnitude(x: float) -> float:
    return abs(float(x))


with workflow_run(seed=4):
    sequential_result = magnitude.using(executor="sequential")(prior)
with workflow_run(seed=4):
    threaded_result = magnitude.using(executor="thread")(prior)

print("result atoms agree:", bool(jnp.allclose(sequential_result.atoms.raw, threaded_result.atoms.raw)))

result atoms agree: True


## Labels and notation

The label names a term in printed output and in the record of how each result was computed.
Labels do not change values or probabilities, so two terms can share one.
A constructor derives a label when its data or structure supplies a description:

1. **a distribution constructed from a family:** the family's name, such as `Normal`;
2. **a record:** its field names;
3. **a function:** the Python function's name, with `𝒻` for a lambda function.

Values holding raw arrays require `label=` because their contents do not identify what they represent.
A callable without a name also requires a label.

In [20]:
theta = Normal("theta", 0.0, 1.0)
weights = NumericArray(jnp.array([0.2, 0.5, 0.3]), label="weights")


@function
def rescale(x: jax.Array) -> jax.Array:
    # Divide by the total, so the entries sum to one.
    return x / jnp.sum(x)


@function
def doubled(x: jax.Array) -> jax.Array:
    return 2.0 * x


print("label of theta:", theta.label)
print("label of weights:", weights.label)
print("label of rescale:", rescale.label)

label of theta: Normal
label of weights: weights
label of rescale: rescale


`label=` sets another label at construction, and `with_label` returns a copy under a new label:

In [21]:
prior = Normal("theta", 0.0, 1.0, label="prior")
renamed = theta.with_label("prior")

print("label set at construction:", prior.label)
print("label of the copy:", renamed.label)
print("label of the original:", theta.label)

label set at construction: prior
label of the copy: prior
label of the original: Normal


A distribution, a conditional distribution, and a function display as their **notation**, which is the label followed by the **signature**.
The signature states what the term is over:

1. **a distribution:** its outputs;
2. **a conditional distribution:** its outputs, then `|` and its inputs;
3. **a function:** its parameters.

`print` and `str()` show the notation, and `repr()` shows the class, the label, and the parameters:

In [22]:
def y_given_theta(theta: jax.Array) -> Normal:
    return Normal("y", theta, 1.0)


likelihood = conditional_distribution(
    y_given_theta, label="likelihood", given_spec={"theta": NumericArraySpec(())}
)

print("notation of prior:", prior)
print("repr of prior:", repr(prior))
print("notation of likelihood:", likelihood)
print("notation of rescale:", rescale)

notation of prior: prior(theta)
repr of prior: Normal('theta', loc=0.0, scale=1.0, label='prior')
notation of likelihood: likelihood(y | theta)
notation of rescale: rescale(x)


`prior` is over `theta`, `likelihood` is over `y` given `theta`, and `rescale` takes the parameter `x`.
`&` joins the labels of its operands with `·`, so a product displays factor by factor, and `with_label` names the product:

In [23]:
model = (likelihood & prior).with_label("model")

print("notation of the product:", (likelihood & prior))
print("notation of the model:", model)

notation of the product: likelihood(y | theta)·prior(theta)
notation of the model: model(y, theta)


An operation labels its result from the terms that the result was computed from, and it labels a value computed from a distribution in probability notation:

1. **a draw:** its outputs, `~`, and the label of the distribution, as `theta ~ prior`;
2. **a log density:** `log` and the notation of the distribution, as `log prior(theta)`;
3. **a mean, a variance, or a quantile:** `𝔼`, `𝕍`, or `ℚ` and a draw in brackets, as `𝔼[theta ~ prior]`;
4. **the result of a function:** a value's label describes the application, as `rescale(weights)` does; a returned distribution has the function's label and displays the application in its notation, as `doubled(theta ~ prior)` does.

In [24]:
with workflow_run(seed=0):
    draw = sample(prior)
    doubled_prior = doubled(prior)

print("label of a draw of prior:", draw.label)
print("label of the log density of prior:", log_prob(prior, 0.5).label)
print("label of the mean of prior:", mean(prior).label)
print("label of rescale(weights):", rescale(weights).label)
print("notation of doubled(prior):", doubled_prior)

label of a draw of prior: theta ~ prior
label of the log density of prior: log prior(theta)
label of the mean of prior: 𝔼[theta ~ prior]


label of rescale(weights): rescale(weights)
notation of doubled(prior): doubled(theta ~ prior)


A conditioned distribution records the paths at which it is conditioned, after `;` in its signature.
Conditioning the model on a value of `theta` gives the likelihood at that value:

In [25]:
likelihood_at_half = condition_on(model, {"theta": 0.5})

print("the model conditioned on theta = 0.5:", likelihood_at_half)

the model conditioned on theta = 0.5: likelihood(y; theta)


The result is the factor `likelihood` at `theta = 0.5`, so it keeps that factor's label, and `; theta` records the conditioned path.
The posterior of `model` given data `y` displays as `model(theta; y)` in the same way, as [Fitting a model to data](06_conditioning_and_inference.ipynb) shows.

A derived label nests the labels it is built from, as `𝔼[theta ~ prior]` nests the draw `theta ~ prior`.
`probpipe.config.notation.max_depth`, which is 8 by default, sets the number of nested levels that a label or a notation shows.
A deeper part shows as its label or as `…`, with a warning, so the label of a long chain of operations, such as a loop that adds to a value, stays short.

## Immutability

A term cannot be changed after construction, so assigning to one of its attributes raises `AttributeError`:

In [26]:
try:
    weights.label = "w"
except AttributeError as error:
    print("assigning a label raised:", error)

assigning a label raised: NumericArray is immutable


A change therefore returns a new term and leaves the original as it was, as `with_label` did above.
[Your data: values, records, and batches](04_values_and_records.ipynb) shows the changes to a record's fields, which return new records in the same way.

A term stores the NumPy array it is given and marks the array read-only, so writing into it afterwards raises `ValueError`:

In [27]:
counts = np.array([3, 5, 2])
count_term = NumericArray(counts, label="counts")

try:
    counts[0] = 4
except ValueError as error:
    print("writing into the stored array raised:", error)

writing into the stored array raised: assignment destination is read-only


To change the values, build a new array and a new term from it.
To keep the original array writable, pass a copy, as in `NumericArray(counts.copy(), label="counts")`.

A term's **annotations** are the one part that can be written after construction.
Its `.annotations` is `None` until a writer attaches results.
Diagnostic functions attach their results after construction, as `add_mcmc_diagnostics` does in [Checking a fit](07_checking_results.ipynb).

## See also

- [Operations](../api/operations.md): `mean`, `sample`, `condition_on`, and the errors `ResolutionError` and `MathematicalDomainError`.
- [Registries for extensions](../api/extending.md): `operation_registry` and the routes of an operation.
- [Workflows and reproducibility](../api/workflows.md): `workflow_run`, `replay_run`, and `config.sampling`.
- [Labels and provenance](../api/provenance.md): the provenance record, `provenance_ancestors`, `provenance_dag`, and the settings of how much history a provenance keeps.
- [Propagating uncertainty through functions](03_functions_and_lifting.ipynb): lifting a function to a distribution.
- [Fitting a model to data](06_conditioning_and_inference.ipynb): the routes and methods of `condition_on`.
- [Tutorial 2](../tutorials/02_fitting.ipynb) and [tutorial 4](../tutorials/04_forecasting.ipynb): the provenance of a fitted posterior and of a forecast.
- [Tutorial 3](../tutorials/03_revising.ipynb): choosing an inference method with `using`, and `exact_only`.